In [1]:
!pip install --upgrade protobuf

  Using cached protobuf-7.35.0-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
Using cached protobuf-7.35.0-cp310-abi3-macosx_10_9_universal2.whl (433 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.14 requires protobuf<5,>=4.25.3, but you have protobuf 7.35.0 which is incompatible.
google-api-core 2.30.1 requires protobuf<7.0.0,>=4.25.8, but you have protobuf 7.35.0 which is incompatible.


In [2]:
!pip install tensorflow

In [3]:
!pip uninstall -y mediapipe
!pip install mediapipe==0.10.14

Found existing installation: mediapipe 0.10.14
Uninstalling mediapipe-0.10.14:
  Successfully uninstalled mediapipe-0.10.14
  Using cached mediapipe-0.10.14-cp312-cp312-macosx_11_0_universal2.whl.metadata (9.7 kB)
  Using cached protobuf-4.25.9-cp37-abi3-macosx_10_9_universal2.whl.metadata (541 bytes)
Using cached mediapipe-0.10.14-cp312-cp312-macosx_11_0_universal2.whl (50.2 MB)
Using cached protobuf-4.25.9-cp37-abi3-macosx_10_9_universal2.whl (394 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.0
    Uninstalling protobuf-7.35.0:
      Successfully uninstalled protobuf-7.35.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 4.25.9 which is incompatible.


In [5]:
import sys
print(sys.executable)

/Users/nghi_nguyen/miniforge3/envs/dsc80/bin/python


In [6]:
import sys

!{sys.executable} -m pip uninstall -y protobuf tensorflow mediapipe
!{sys.executable} -m pip install protobuf==5.28.3
!{sys.executable} -m pip install tensorflow mediapipe opencv-python numpy

Found existing installation: protobuf 4.25.9
Uninstalling protobuf-4.25.9:
  Successfully uninstalled protobuf-4.25.9
Found existing installation: tensorflow 2.21.0
Uninstalling tensorflow-2.21.0:
  Successfully uninstalled tensorflow-2.21.0
Found existing installation: mediapipe 0.10.14
Uninstalling mediapipe-0.10.14:
  Successfully uninstalled mediapipe-0.10.14
  Using cached tensorflow-2.21.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (4.4 kB)
  Using cached mediapipe-0.10.35-py3-none-macosx_11_0_arm64.whl.metadata (9.8 kB)
  Using cached protobuf-7.35.0-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
Using cached tensorflow-2.21.0-cp312-cp312-macosx_12_0_arm64.whl (223.5 MB)
Using cached mediapipe-0.10.35-py3-none-macosx_11_0_arm64.whl (17.8 MB)
Using cached protobuf-7.35.0-cp310-abi3-macosx_10_9_universal2.whl (433 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.28.3
    Uninstalling protobuf-5.28.3:
      Successfully uninstalled protob

In [7]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import time
from collections import deque

NameError: name 'core' is not defined

In [ ]:
# Load trained model and normalization files
model = tf.keras.models.load_model("best_tcn_gesture_model.keras")

mean = np.load("landmark_mean.npy")
std = np.load("landmark_std.npy")

print("Model loaded!")
print("Mean shape:", mean.shape)
print("Std shape:", std.shape)

In [ ]:
# Constants
SEQUENCE_LENGTH = 37
NUM_LANDMARKS = 21
FEATURES_PER_FRAME = NUM_LANDMARKS * 3  # 63

# Rolling buffer stores the most recent 37 frames of landmarks
landmark_buffer = deque(maxlen=SEQUENCE_LENGTH)

In [ ]:
# Placeholder class mapping
# Later, replace this with the actual label_id → gesture name mapping
class_names = {
    0: "Class 0",
    1: "Class 1",
    2: "Class 2",
    3: "Class 3",
    4: "Class 4",
    5: "Class 5",
    6: "Class 6",
    7: "Class 7",
    8: "Class 8",
    9: "Class 9",
    10: "Class 10",
    11: "Class 11",
    12: "Class 12",
    13: "Class 13",
    14: "Class 14",
    15: "Class 15",
    16: "Class 16",
    17: "Class 17",
    18: "Class 18",
    19: "Class 19",
    20: "Class 20",
    21: "Class 21",
    22: "Class 22",
    23: "Class 23",
    24: "Class 24",
    25: "Class 25",
    26: "Class 26",
}

In [ ]:
def extract_landmarks_from_mediapipe(hand_landmarks):
    """
    Converts MediaPipe hand landmarks into a flat vector of 63 values.
    Output shape: (63,)
    """
    landmarks = []

    for lm in hand_landmarks.landmark:
        landmarks.extend([lm.x, lm.y, lm.z])

    return np.array(landmarks, dtype=np.float32)

In [ ]:
def predict_gesture(landmark_sequence):
    """
    landmark_sequence shape: (37, 63)
    Returns predicted class index and confidence.
    """

    x = landmark_sequence.astype("float32")

    # Reshape mean/std safely
    mean_fixed = np.squeeze(mean)
    std_fixed = np.squeeze(std)

    # Normalize same way as training
    x = (x - mean_fixed) / std_fixed

    # Add batch dimension: (1, 37, 63)
    x = np.expand_dims(x, axis=0)

    prediction = model.predict(x, verbose=0)

    class_idx = int(np.argmax(prediction, axis=1)[0])
    confidence = float(np.max(prediction))

    return class_idx, confidence

In [ ]:
# MediaPipe setup
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    max_num_hands=1,  # use 1 hand because your model was trained on 1 hand
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)